<a href="https://colab.research.google.com/github/saumya758/ai-vs-real-image-detection/blob/main/phase3_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install ftfy regex tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00


In [ ]:
!pip install openai-clip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for openai-clip: filename=openai_clip-1.0.1-py3-none-any.whl size=1368605 sha256=7fb1f96b4ed0a6a27cc256205447fd1ae83d04166b79653d89b5f7391d09dbda
  Stored in directory: /root/.cache/pip/wheels/ab/49/bc/c2342e8e14878210ba4825cf314a53f2570f6fb18b91fce3cf
Successfully built openai-clip


In [ ]:
#load model
import clip

clip.available_models()

['RN50',
 'RN101',
 'RN50x4',
 'RN50x16',
 'RN50x64',
 'ViT-B/32',
 'ViT-B/16',
 'ViT-L/14',
 'ViT-L/14@336px']

In [ ]:
import numpy as np

model, preprocess = clip.load("ViT-B/32")
model.cuda().eval()
input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 148MiB/s]


Model parameters: 151,277,313
Input resolution: 224
Context length: 77
Vocab size: 49408


In [ ]:
!pip install ftfy regex tqdm
!pip install openai-clip

import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

print("CLIP loaded successfully on", device)

ModuleNotFoundError: No module named 'model'

In [ ]:

import clip

def load_model():
    print("Loading model...")
    return model, preprocess

In [ ]:

from PIL import Image

def load_image(image_path):
    # Placeholder for loading an image
    print(f"Loading image from {image_path}...")
    try:
        image = Image.open(image_path).convert("RGB")
        return image
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return None

In [ ]:
# image preprocessing
preprocess

In [ ]:
#text preprocessing
clip.tokenize("Hello World!")

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import torch
import clip
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

uploaded = files.upload()

original_images = []
images = []

for idx, img_path in enumerate(uploaded.keys()):
    image = Image.open(img_path).convert("RGB")

    plt.imshow(image)
    plt.title("Uploaded Image")
    plt.axis("off")
    plt.show()

    original_images.append(image)

    image_input = preprocess(image).unsqueeze(0).to(device)
    images.append(image_input)

In [ ]:
texts = [
    "a real natural photograph",
    "an ai generated image"
]

text_tokens = clip.tokenize(texts).to(device)

In [ ]:
image_input = torch.cat(images, dim=0).to(device)

In [ ]:
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_tokens)

image_features = image_features / image_features.norm(dim=-1, keepdim=True)
text_features = text_features / text_features.norm(dim=-1, keepdim=True)

similarity = (image_features @ text_features.T).softmax(dim=-1)

In [ ]:
pred_idx = similarity[0].argmax().item()
confidence = similarity[0][pred_idx].item()

print("Prediction:", texts[pred_idx])
print("Confidence:", round(confidence * 100, 2), "%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
similarity_cpu = similarity.detach().cpu().numpy()

num_processed_images = similarity_cpu.shape[0]

plt.figure(figsize=(10, 6 * num_processed_images))

for i in range(num_processed_images):
    image = original_images[i]
    plt.subplot(num_processed_images, 2, 2 * i + 1)
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Uploaded Image {i+1}")

    plt.subplot(num_processed_images, 2, 2 * i + 2)

    y = np.arange(len(texts))
    plt.barh(y, similarity_cpu[i], color='skyblue')

    plt.gca().invert_yaxis()
    plt.yticks(y, texts)
    plt.xlabel("Probability")

    pred_idx = similarity_cpu[i].argmax()
    confidence = similarity_cpu[i][pred_idx]

    plt.title(f"Prediction: {texts[pred_idx]} ({confidence*100:.2f}%)")

    for j, val in enumerate(similarity_cpu[i]):
        plt.text(val, j, f"{val:.2f}", va='center')

plt.tight_layout()
plt.show()

In [ ]:

similarity_cpu = similarity.detach().cpu().numpy()
similarity_plot_matrix = similarity_cpu.T
num_texts_for_plot = similarity_plot_matrix.shape[0]
num_images_for_plot = similarity_plot_matrix.shape[1]

import matplotlib.pyplot as plt

plt.figure(figsize=(20, 14))
plt.imshow(similarity_plot_matrix, vmin=0.1, vmax=0.3)
plt.yticks(range(num_texts_for_plot), texts, fontsize=18)
plt.xticks(range(num_images_for_plot), [f"Img {i+1}" for i in range(num_images_for_plot)], fontsize=14)

for i, image in enumerate(original_images[:num_images_for_plot]):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

for x_idx in range(num_images_for_plot):
    for y_idx in range(num_texts_for_plot):
        plt.text(x_idx, y_idx, f"{similarity_plot_matrix[y_idx, x_idx]:.2f}",
                 ha="center", va="center", size=12)

for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, num_images_for_plot - 0.5])
plt.ylim([num_texts_for_plot + 0.5, -2])

plt.title("Cosine similarity between text and image features", size=20)
plt.colorbar()

plt.show()

In [ ]:
#Zero-Shot Image Classification
import os
from torchvision.datasets import CIFAR100

cifar100 = CIFAR100(os.path.expanduser("~/.cache"), transform=preprocess, download=True)

In [ ]:
text_descriptions = [f"This is a photo of a {label}" for label in cifar100.classes]
text_tokens = clip.tokenize(text_descriptions).cuda()

In [ ]:
with torch.no_grad():
    text_features = model.encode_text(text_tokens).float()
    text_features /= text_features.norm(dim=-1, keepdim=True)

text_probs = (100.0 * image_features.float() @ text_features.T).softmax(dim=-1)
top_probs, top_labels = text_probs.cpu().topk(5, dim=-1)

In [ ]:
plt.figure(figsize=(16, 16))

for i, image in enumerate(original_images):
    plt.subplot(4, 4, 2 * i + 1)
    plt.imshow(image)
    plt.axis("off")

    plt.subplot(4, 4, 2 * i + 2)
    y = np.arange(top_probs.shape[-1])
    plt.grid()
    plt.barh(y, top_probs[i])
    plt.gca().invert_yaxis()
    plt.gca().set_axisbelow(True)
    plt.yticks(y, [cifar100.classes[index] for index in top_labels[i].numpy()])
    plt.xlabel("probability")

plt.subplots_adjust(wspace=0.5)
plt.show()